# Day 5 — Space Complexity Lab

Yesterday we measured TIME. Today the other half — MEMORY. Same Big-O language, different resource.

**How to use this notebook:** every experiment says PREDICT first. Write your guess on paper, THEN run the cell. A wrong guess that you thought about teaches more than ten right guesses you didn't.

## 1. Recursion = a stack of plates

Each unfinished call waits on the **call stack**, like plates in the hostel mess — last plate on, first plate off.

**PREDICT:** what exact order do the lines print for `countdown(3)`? When does "Blast off!" appear — first or last? And what prints AFTER it?

In [ ]:
def countdown(n):
    if n == 0:                      # base case — the stopping plate
        print("Blast off!")
        return
    print("going down:", n)
    countdown(n - 1)                # this call WAITS here
    print("coming back up:", n)     # runs only AFTER the inner call finishes

countdown(3)
# The "coming back up" lines prove the waiting plates existed:
# they run in REVERSE order, as plates come off the top of the stack.

At the deepest moment, 4 calls were alive at once → 4 plates → **recursion space = O(max depth)**, even with zero lists created.

## 2. The plate-stack ceiling

Python refuses to stack plates forever.

**PREDICT:** what number does `sys.getrecursionlimit()` print? And what happens if we try to go deeper than that?

In [ ]:
import sys
print("Python's plate-stack limit:", sys.getrecursionlimit())

def deep(n):
    if n == 0:
        return "reached the bottom!"
    return deep(n - 1)

print(deep(900))          # fits under the ceiling — fine

try:
    deep(5000)            # too many plates!
except RecursionError as e:
    print("CRASH at ~1000 deep:", e)

Lesson: the logic of `deep(5000)` is perfectly correct — the **memory ceiling** killed it. For deep work in Python, prefer a loop: a loop reuses ONE frame → O(1) auxiliary space.

## 3. Reverse: copy vs in-place (and the `[::-1]` trap)

**PREDICT:** after each reversal below, is the ORIGINAL list changed or untouched? And is `arr[::-1]` in-place or a copy? The `id()` function (a value's memory address) will tell the truth.

In [ ]:
# Way 1: slicing one-liner
arr = [1, 2, 3, 4, 5]
rev = arr[::-1]
print("original:", arr, "| reversed:", rev)
print("same object in memory?", id(arr) == id(rev))   # False → a COPY was built → O(n) space

# Way 2: two-pointer, in-place
arr2 = [1, 2, 3, 4, 5]
left, right = 0, len(arr2) - 1
while left < right:
    arr2[left], arr2[right] = arr2[right], arr2[left]
    left += 1
    right -= 1
print("in-place result:", arr2, "— same object, O(1) auxiliary space")
# Trade-off: no extra memory, but the ORIGINAL order is gone.

## 4. String `+=` vs `join` — the reprinted wedding card

Strings are immutable: `s += ch` reprints the whole card every time.

**PREDICT:** for 100,000 characters, roughly how many times slower is `+=` than `join`? 2x? 10x? more?

In [ ]:
import time

n = 100_000

start = time.perf_counter()
s = ""
for i in range(n):
    s += "x"                      # full reprint each time → O(n²)
t_plus = time.perf_counter() - start

start = time.perf_counter()
pieces = []
for i in range(n):
    pieces.append("x")            # O(1) each
s2 = "".join(pieces)              # one O(n) pass
t_join = time.perf_counter() - start

print(f"+= loop : {t_plus:.4f} s")
print(f"join way: {t_join:.4f} s")
print(f"join is about {t_plus / t_join:.0f}x faster here")
# Try n = 1_000_000 and watch the gap explode — that is O(n²) vs O(n).

## 5. `insert(0)` vs `append` vs `deque` — the train-berth shuffle

`insert(0, x)` makes every passenger shift one seat. `append` takes the free last seat. `deque.appendleft` has a door at BOTH ends.

**PREDICT:** rank the three loops below from fastest to slowest for 50,000 items.

In [ ]:
import time
from collections import deque

n = 50_000

start = time.perf_counter()
a = []
for i in range(n):
    a.insert(0, i)                # everyone shifts → O(n) each → O(n²) total
t_insert = time.perf_counter() - start

start = time.perf_counter()
b = []
for i in range(n):
    b.append(i)                   # last seat free → O(1) each
b.reverse()                       # one O(n) pass gives the same order
t_append = time.perf_counter() - start

start = time.perf_counter()
d = deque()
for i in range(n):
    d.appendleft(i)               # front door exists → O(1) each
t_deque = time.perf_counter() - start

print(f"insert(0) loop : {t_insert:.4f} s   <- the O(n²) trap")
print(f"append+reverse : {t_append:.4f} s")
print(f"deque.appendleft: {t_deque:.4f} s")
print("same result?", a == b == list(d))

## 6. `in` on a list vs a set — door-knocking vs the hotel register

**PREDICT:** 2,000 membership checks against 100,000 items. How much faster is the set — 10x? 100x? 1000x?

In [ ]:
import time

big_list = list(range(100_000))
big_set = set(big_list)           # one-time O(n) conversion — the space we SPEND

start = time.perf_counter()
found = 0
for x in range(0, 200_000, 100):          # 2000 lookups, half will miss
    if x in big_list:                      # door-to-door → O(n) each
        found += 1
t_list = time.perf_counter() - start

start = time.perf_counter()
found2 = 0
for x in range(0, 200_000, 100):
    if x in big_set:                       # reception register → ~O(1) each
        found2 += 1
t_set = time.perf_counter() - start

print(f"list lookups: {t_list:.4f} s")
print(f"set  lookups: {t_set:.6f} s")
print(f"set is about {t_list / t_set:.0f}x faster — bought with O(n) extra memory")
print("same answers?", found == found2)

That is the **space-for-time principle**: spend memory (the set copy) to buy speed. The in-place reverse in section 3 was the opposite trade.

## 7. `chr()` and `ord()` — today's new pattern tool

Every character has a code number (ASCII). `chr(code)` → character, `ord(char)` → code.

**PREDICT:** what is `chr(65)`? What is `ord('E') - ord('A')`?

In [ ]:
print("ord('A') =", ord('A'))
print("chr(65)  =", chr(65))
print("ord('E') - ord('A') =", ord('E') - ord('A'))

# The i-th capital letter is chr(65 + i):
for i in range(5):
    print(i, "->", chr(65 + i), "(code", 65 + i, ")")

# One row of a letter pattern, just to feel the tool (NOT a solution):
print("".join(chr(65 + j) for j in range(5)))   # ABCDE

## 8. Patterns 16–22 — shapes + hints (solve in `main.py`, on paper first!)

**Pattern 16 — Alphabet-repeat triangle**
```
A
BB
CCC
DDDD
EEEEE
```
Hint: row `i` fixes ONE letter `chr(65 + i)` and repeats it `i + 1` times.

**Pattern 17 — Alphabet hill**
```
    A
   ABA
  ABCBA
 ABCDCBA
ABCDEDCBA
```
Hint: `n - i - 1` spaces, climb `A` up to the i-th letter, walk back down WITHOUT repeating the peak.

**Pattern 18 — Reverse-alphabet triangle**
```
E
D E
C D E
B C D E
A B C D E
```
Hint: every row ends at `chr(64 + n)`; row `i` starts `i` letters earlier — start code `(64 + n) - i`.

**Pattern 19 — Hourglass of stars**
```
**********
****  ****
***    ***
**      **
*        *
*        *
**      **
***    ***
****  ****
**********
```
Hint: top half row `i` = `n - i` stars, `2i` spaces, `n - i` stars; bottom half = same rows reversed.

**Pattern 20 — Butterfly**
```
*        *
**      **
***    ***
****  ****
**********
****  ****
***    ***
**      **
*        *
```
Hint: mirror of 19 — with `i` stars per side, the gap is `2 * (n - i)` spaces; `2n - 1` rows total.

**Pattern 21 — Hollow rectangle**
```
*****
*   *
*   *
*   *
*****
```
Hint: star only on a border — first/last row OR first/last column — else a space.

**Pattern 22 — Number rings**
```
5 5 5 5 5 5 5 5 5
5 4 4 4 4 4 4 4 5
5 4 3 3 3 3 3 4 5
5 4 3 2 2 2 3 4 5
5 4 3 2 1 2 3 4 5
5 4 3 2 2 2 3 4 5
5 4 3 3 3 3 3 4 5
5 4 4 4 4 4 4 4 5
5 5 5 5 5 5 5 5 5
```
Hint: `(2n-1) × (2n-1)` grid; cell value = `n - min(i, j, size-1-i, size-1-j)` — n minus distance to the nearest edge.

## 9. Maths problems — hints only (your Day-1 `%` and `//` friends return)

**Trailing zeroes in n!** — do NOT compute n!. Each zero = one 10 = one (2 × 5) pair; 2s are everywhere, 5s are rare → count the 5s: `n//5 + n//25 + n//125 + ...` until 0. Why `//25` too? 25 = 5 × 5 carries TWO fives but `n//5` counted only one. Check: n = 25 → 5 + 1 = 6 zeroes.

**Digit sum** — `n % 10` peels the last digit, `n // 10` removes it; loop while `n > 0`. 5341 → 534 → 53 → 5 → 0. Time = O(number of digits) = O(log₁₀ n).

**Count digits WITHOUT a loop** — way 1: `len(str(n))` (mind the minus sign). Way 2: `floor(log10(n)) + 1` — digit count jumps exactly at powers of 10, and log10 tells you which power-band `n` sits in. Edge cases: n = 0 (answer 1) and negatives (`abs` first).

Now go attempt everything in `main.py`. Tomorrow: the **list**, our first real data structure — where every cost you priced today becomes daily practice.